# Fairness-Aware Property Assessment Modeling in Cook County

# PART I: Problem, Data, and Fairness Framework

## 1. Project Motivation

Property valuation models affect more than predictive accuracy. In a property-tax system, systematic overvaluation or undervaluation can translate into unequal tax burdens across homeowners and communities.

A model may achieve strong aggregate predictive performance while producing systematically different errors for different property-value or demographic groups. Therefore, evaluating a property assessment model requires examining both:

1. **Predictive accuracy** — how closely predicted property values match observed sale prices.
2. **Assessment fairness** — whether the magnitude and direction of prediction errors differ systematically across groups.

This project develops and evaluates property valuation models using residential property sales from Cook County, Illinois. In addition to property characteristics, Census tract-level demographic information will be incorporated to examine whether model errors differ across socioeconomic and demographic contexts.

The project will ultimately compare:

- a conventional linear-regression baseline,
- a stronger predictive benchmark,
- and a fairness-aware model that explicitly penalizes undesirable directional assessment errors.

The central research question is:

> **Can a property valuation model maintain competitive predictive accuracy while reducing systematic assessment disparities across property-value and demographic groups?**

## 2. What Does "Fair" Mean in This Project?

Fairness in property assessment is not equivalent to requiring every individual prediction to be correct.

Instead, we are interested in whether prediction errors exhibit systematic patterns across groups.

For a property with observed sale price $y_i$ and predicted value $\hat{y}_i$:

- $ \hat{y}_i > y_i\ $ represents **overassessment**
- $ \hat{y}_i < y_i\ $ represents **underassessment**

A model may have reasonable overall RMSE while still disproportionately overassessing one segment of the housing market.

We will therefore evaluate models along several dimensions:

### Predictive Performance
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- Mean Absolute Percentage Error (MAPE)

### Directional Assessment Behavior
- Mean residual
- Overassessment rate
- Underassessment rate
- Assessment ratio

$$
\text{Assessment Ratio}_i = \frac{\hat{y}_i}{y_i}
$$

An assessment ratio greater than 1 indicates overassessment, while a ratio below 1 indicates underassessment.

### Group-Level Fairness

These quantities will later be compared across:

1. Property-value groups
2. Census tract income groups
3. Census tract demographic composition
4. Geographic areas

The goal is not to assume that property value itself represents race or income. Instead, demographic conclusions will only be made after explicitly joining external Census/ACS demographic data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os
import requests

In [2]:
with zipfile.ZipFile('cook_county_data.zip', 'r') as z:
    with z.open("cook_county_train.csv") as f:
        data = pd.read_csv(f)

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (204792, 63)


,Unnamed: 0,PIN,Property Class,Neighborhood Code,Land Square Feet,Town Code,Apartments,Wall Material,Roof Material,Basement,...,Sale Month of Year,Sale Half of Year,Most Recent Sale,Age Decade,Pure Market Filter,Garage Indicator,Neigborhood Code (mapping),Town and Neighborhood,Description,Lot Size
0,0,17294100610000,203,50,2500.0,76,0.0,2.0,1.0,1.0,...,9,2,1.0,13.2,0,0.0,50,7650,"This property, sold on 09/14/2015, is a one-st...",2500.0
1,1,13272240180000,202,120,3780.0,71,0.0,2.0,1.0,1.0,...,5,1,1.0,9.6,1,1.0,120,71120,"This property, sold on 05/23/2018, is a one-st...",3780.0
2,2,25221150230000,202,210,4375.0,70,0.0,2.0,1.0,2.0,...,2,1,0.0,11.2,1,1.0,210,70210,"This property, sold on 02/18/2016, is a one-st...",4375.0
3,3,10251130030000,203,220,4375.0,17,0.0,3.0,1.0,1.0,...,7,2,1.0,6.3,1,1.0,220,17220,"This property, sold on 07/23/2013, is a one-st...",4375.0
4,4,31361040550000,202,120,8400.0,32,0.0,3.0,1.0,2.0,...,6,1,0.0,6.3,1,1.0,120,32120,"This property, sold on 06/10/2016, is a one-st...",8400.0


In [3]:
print(f"Number of observations: {data.shape[0]:,}")
print(f"Number of columns: {data.shape[1]}")

data.info()

Number of observations: 204,792
Number of columns: 63
<class 'pandas.DataFrame'>
RangeIndex: 204792 entries, 0 to 204791
Data columns (total 63 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Unnamed: 0                  204792 non-null  int64  
 1   PIN                         204792 non-null  int64  
 2   Property Class              204792 non-null  int64  
 3   Neighborhood Code           204792 non-null  int64  
 4   Land Square Feet            204792 non-null  float64
 5   Town Code                   204792 non-null  int64  
 6   Apartments                  204792 non-null  float64
 7   Wall Material               204792 non-null  float64
 8   Roof Material               204792 non-null  float64
 9   Basement                    204792 non-null  float64
 10  Basement Finish             204792 non-null  float64
 11  Central Heating             204792 non-null  float64
 12  Other Heating               2

## 4. Core Variables

The raw dataset contains property characteristics, transaction information, geographic identifiers, and assessment-related variables.

Before performing exploratory analysis, we first identify the variables most relevant to the project:

### Outcome
- `Sale Price`

### Property Characteristics
Examples include:
- `Building Square Feet`
- `Land Square Feet`
- `Bathrooms`
- `Age`
- `Property Class`
- construction and condition variables
- garage and improvement characteristics

### Geography
- `Census Tract`
- `Latitude`
- `Longitude`
- `Town Code`
- `Neighborhood Code`

### Time
- `Sale Year`
- `Sale Quarter`
- `Sale Month of Year`

The `Census Tract` variable is especially important because it potentially provides a direct key for joining tract-level demographic information from the American Community Survey.

In [4]:
{
    "shape": data.shape,
    "sale_year_range": (
        data["Sale Year"].min(),
        data["Sale Year"].max()
    ),
    "n_census_tracts": data["Census Tract"].nunique(),
    "missing_census_tract_pct": data["Census Tract"].isna().mean() * 100,
    "missing_latitude_pct": data["Latitude"].isna().mean() * 100,
    "missing_longitude_pct": data["Longitude"].isna().mean() * 100,
}

{'shape': (204792, 63),
 'sale_year_range': (np.int64(2013), np.int64(2019)),
 'n_census_tracts': 1265,
 'missing_census_tract_pct': np.float64(0.0),
 'missing_latitude_pct': np.float64(0.0),
 'missing_longitude_pct': np.float64(0.0)}

In [5]:
data[
    [
        "Census Tract",
        "Latitude",
        "Longitude",
        "Town Code",
        "Neighborhood Code",
        "Sale Year",
        "Sale Price"
    ]
].head(10)

,Census Tract,Latitude,Longitude,Town Code,Neighborhood Code,Sale Year,Sale Price
0,600600.0,41.840803,-87.654264,76,50,2015,1
1,200100.0,41.933016,-87.735966,71,120,2018,285000
2,491400.0,41.686738,-87.616496,70,210,2016,22000
3,810301.0,42.019937,-87.701482,17,220,2013,225000
4,830300.0,41.476430,-87.682236,32,120,2016,22600
5,640300.0,41.784580,-87.769661,72,380,2018,1
6,828202.0,41.557900,-87.547981,37,181,2017,100000
7,801606.0,42.118618,-87.855616,25,52,2016,795000
8,160800.0,41.953313,-87.712016,71,70,2016,675000
9,804105.0,42.091755,-88.101858,29,33,2019,270000


### Examining Data Structure/Quality

The raw Cook County dataset contains **204792 property-sale observations and 63 variables**, covering transactions from **2013 through 2019**.

Geographic coverage is strong for the planned demographic analysis:

- The dataset contains **1,265 unique Census tracts**.
- Census tract, latitude, and longitude information are complete, with **0% missingness**.
- Therefore, tract-level demographic enrichment appears feasible without discarding a meaningful portion of the housing dataset.

The initial inspection also reveals transactions recorded at extremely small sale prices, including \$1 sales. These likely represent non-market or nominal transactions and should be investigated during the data-cleaning stage rather than treated as ordinary residential sales.

## 5. Census Geography Validation

To incorporate demographic information, each property must be linked to a Census tract.

The Census Bureau identifies Census tracts using a six-digit tract code within a state and county. For Cook County:

- Illinois state FIPS code: `17`
- Cook County FIPS code: `031`

The housing dataset stores Census tract identifiers numerically, which may remove leading zeroes. We therefore standardize the tract variable as a six-character string before attempting the demographic join.

The complete tract GEOID can then be represented as:

$$
\text{GEOID} =
\text{State FIPS} +
\text{County FIPS} +
\text{Tract Code}
$$

For Cook County, this gives an 11-digit identifier beginning with `17031`.

In [6]:
# Preserve missing values while creating a standardized six-digit tract code
data["Census Tract Code"] = (
    data["Census Tract"]
    .astype("Int64")
    .astype("string")
    .str.zfill(6)
)

# Construct full 11-digit Census tract GEOID
data["Census Tract GEOID"] = (
    "17031" + data["Census Tract Code"]
)

data[
    [
        "Census Tract",
        "Census Tract Code",
        "Census Tract GEOID"
    ]
].head(10)

,Census Tract,Census Tract Code,Census Tract GEOID
0,600600.0,600600,17031600600
1,200100.0,200100,17031200100
2,491400.0,491400,17031491400
3,810301.0,810301,17031810301
4,830300.0,830300,17031830300
5,640300.0,640300,17031640300
6,828202.0,828202,17031828202
7,801606.0,801606,17031801606
8,160800.0,160800,17031160800
9,804105.0,804105,17031804105


In [7]:
print("Unique raw tracts:", data["Census Tract"].nunique())
print("Unique standardized tracts:", data["Census Tract Code"].nunique())

print("\nTract-code length distribution:")
print(data["Census Tract Code"].str.len().value_counts(dropna=False))

print("\nSample standardized tracts:")
print(
    data[
        ["Census Tract", "Census Tract Code", "Census Tract GEOID"]
    ]
    .drop_duplicates()
    .sort_values("Census Tract Code")
    .head(15)
)

Unique raw tracts: 1265
Unique standardized tracts: 1265

Tract-code length distribution:
Census Tract Code
6    204792
Name: count, dtype: Int64

Sample standardized tracts:
        Census Tract Census Tract Code Census Tract GEOID
6603         10100.0            010100        17031010100
2679         10201.0            010201        17031010201
2644         10202.0            010202        17031010202
9933         10300.0            010300        17031010300
14568        10400.0            010400        17031010400
63862        10501.0            010501        17031010501
40883        10502.0            010502        17031010502
179851       10503.0            010503        17031010503
2877         10600.0            010600        17031010600
816          10701.0            010701        17031010701
10303        10702.0            010702        17031010702
18515        20100.0            020100        17031020100
237          20200.0            020200        17031020200
1696         

### Census Tract Identifier Validation

Standardizing the Census tract variable successfully preserved all **1,265 unique tract identifiers** while converting each non-missing value to the six-digit format used by the Census Bureau.

Of the 204792 property-sale observations, **all have a valid six-digit tract code**, with only one observation missing geographic information. The resulting 11-digit GEOID combines the Illinois state FIPS (`17`), Cook County FIPS (`031`), and six-digit tract code.

Because no tract identifiers were lost or duplicated during standardization, the resulting GEOID is a suitable candidate key for joining external tract-level demographic data.

## 6. Validate Property Tracts Against Official Census Geography

Before attaching demographic variables, the locally constructed Census GEOIDs should be compared against an official list of Census tracts.

This validation serves two purposes:

1. It verifies that the housing dataset's tract coding is compatible with Census geography.
2. It allows us to measure the expected demographic-data match rate before performing the actual merge.

We use the 2019 ACS 5-year geography because the housing transactions span 2013–2019 and the 2019 release provides a reasonable demographic snapshot near the end of the study period.

In [8]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

print("Key loaded:", CENSUS_API_KEY is not None)

Key loaded: True


In [9]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

url = "https://api.census.gov/data/2019/acs/acs5"

params = {
    "get": "NAME,B01001_001E",
    "for": "tract:*",
    "in": "state:17 county:031",
    "key": CENSUS_API_KEY
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Final URL:", response.url)

response.raise_for_status()

Status code: 200
Content type: application/json;charset=utf-8
Final URL: https://api.census.gov/data/2019/acs/acs5?get=NAME%2CB01001_001E&for=tract%3A%2A&in=state%3A17+county%3A031&key=03e2116ac4a67ea4e0bdaca854a375d654a043bb


In [10]:
acs_raw = response.json()

acs_tracts = pd.DataFrame(
    acs_raw[1:],
    columns=acs_raw[0]
)

acs_tracts["Census Tract GEOID"] = (
    acs_tracts["state"]
    + acs_tracts["county"]
    + acs_tracts["tract"]
)

print("Number of official Cook County ACS tracts:", len(acs_tracts))
acs_tracts.head()

Number of official Cook County ACS tracts: 1319


,NAME,B01001_001E,state,county,tract,Census Tract GEOID
0,"Census Tract 6302, Cook County, Illinois",1825,17,031,630200,17031630200
1,"Census Tract 5807, Cook County, Illinois",5908,17,031,580700,17031580700
2,"Census Tract 5906, Cook County, Illinois",3419,17,031,590600,17031590600
3,"Census Tract 6007, Cook County, Illinois",2835,17,031,600700,17031600700
4,"Census Tract 6119, Cook County, Illinois",1639,17,031,611900,17031611900


In [11]:
property_geoids = set(
    data["Census Tract GEOID"].dropna()
)

acs_geoids = set(
    acs_tracts["Census Tract GEOID"]
)

matched_geoids = property_geoids & acs_geoids
unmatched_geoids = property_geoids - acs_geoids

tract_match_rate = (
    len(matched_geoids) / len(property_geoids) * 100
)

property_match_rate = (
    data["Census Tract GEOID"].isin(acs_geoids).mean() * 100
)

print("Unique property tracts:", len(property_geoids))
print("Matched unique tracts:", len(matched_geoids))
print("Unmatched unique tracts:", len(unmatched_geoids))
print(f"Tract-level match rate: {tract_match_rate:.2f}%")
print(f"Property-level match rate: {property_match_rate:.2f}%")

print("\nSample unmatched tract GEOIDs:")
print(sorted(unmatched_geoids)[:20])

Unique property tracts: 1265
Matched unique tracts: 1265
Unmatched unique tracts: 0
Tract-level match rate: 100.00%
Property-level match rate: 100.00%

Sample unmatched tract GEOIDs:
[]


## 7. Constructing Census-Based Fairness Variables

The successful GEOID validation shows that all census tracts represented in the
Cook County property dataset can be linked to the 2019 ACS 5-Year Estimates.

The next step is to construct tract-level demographic and socioeconomic
variables that can later be used to evaluate whether prediction errors are
systematically distributed across different communities.

These variables are initially treated as **fairness evaluation attributes**
rather than housing-price predictors. This distinction allows the predictive
model to rely primarily on property characteristics while demographic context
is used to examine whether model errors disproportionately affect particular
neighborhoods.

The first set of ACS measures captures:

- population,
- median household income,
- poverty,
- racial and ethnic composition.

In [12]:
acs_variables = {
    # Population
    "B01001_001E": "total_population",

    # Median household income
    "B19013_001E": "median_household_income",

    # Poverty
    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",

    # Race / ethnicity
    "B03002_001E": "race_ethnicity_total",
    "B03002_003E": "white_non_hispanic",
    "B03002_004E": "black_non_hispanic",
    "B03002_006E": "asian_non_hispanic",
    "B03002_012E": "hispanic"
}

params = {
    "get": "NAME," + ",".join(acs_variables.keys()),
    "for": "tract:*",
    "in": "state:17 county:031",
    "key": CENSUS_API_KEY
}

response = requests.get(
    "https://api.census.gov/data/2019/acs/acs5",
    params=params,
    timeout=30
)

print("Status code:", response.status_code)
response.raise_for_status()

Status code: 200


In [13]:
acs_raw = response.json()

acs_fairness = pd.DataFrame(
    acs_raw[1:],
    columns=acs_raw[0]
)

acs_fairness = acs_fairness.rename(columns=acs_variables)

acs_fairness["Census Tract GEOID"] = (
    acs_fairness["state"]
    + acs_fairness["county"]
    + acs_fairness["tract"]
)

acs_fairness.head()

,NAME,total_population,median_household_income,poverty_universe,below_poverty,race_ethnicity_total,white_non_hispanic,black_non_hispanic,asian_non_hispanic,hispanic,state,county,tract,Census Tract GEOID
0,"Census Tract 6302, Cook County, Illinois",1825,37422,1825,436,1825,125,0,78,1622,17,031,630200,17031630200
1,"Census Tract 5807, Cook County, Illinois",5908,47000,5908,1216,5908,423,161,522,4742,17,031,580700,17031580700
2,"Census Tract 5906, Cook County, Illinois",3419,46033,3416,404,3419,757,9,431,2119,17,031,590600,17031590600
3,"Census Tract 6007, Cook County, Illinois",2835,45294,2835,432,2835,1001,82,857,850,17,031,600700,17031600700
4,"Census Tract 6119, Cook County, Illinois",1639,24507,1639,766,1639,26,1175,0,438,17,031,611900,17031611900


In [14]:
numeric_cols = list(acs_variables.values())

for col in numeric_cols:
    acs_fairness[col] = pd.to_numeric(
        acs_fairness[col],
        errors="coerce"
    )

In [15]:
acs_fairness["poverty_rate"] = (
    acs_fairness["below_poverty"]
    / acs_fairness["poverty_universe"]
)

acs_fairness["pct_white_non_hispanic"] = (
    acs_fairness["white_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_black_non_hispanic"] = (
    acs_fairness["black_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_asian_non_hispanic"] = (
    acs_fairness["asian_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_hispanic"] = (
    acs_fairness["hispanic"]
    / acs_fairness["race_ethnicity_total"]
)
acs_fairness.head()

,NAME,total_population,median_household_income,poverty_universe,below_poverty,race_ethnicity_total,white_non_hispanic,black_non_hispanic,asian_non_hispanic,hispanic,state,county,tract,Census Tract GEOID,poverty_rate,pct_white_non_hispanic,pct_black_non_hispanic,pct_asian_non_hispanic,pct_hispanic
0,"Census Tract 6302, Cook County, Illinois",1825,37422,1825,436,1825,125,0,78,1622,17,031,630200,17031630200,0.238904,0.068493,0.000000,0.042740,0.888767
1,"Census Tract 5807, Cook County, Illinois",5908,47000,5908,1216,5908,423,161,522,4742,17,031,580700,17031580700,0.205823,0.071598,0.027251,0.088355,0.802640
2,"Census Tract 5906, Cook County, Illinois",3419,46033,3416,404,3419,757,9,431,2119,17,031,590600,17031590600,0.118267,0.221410,0.002632,0.126060,0.619772
3,"Census Tract 6007, Cook County, Illinois",2835,45294,2835,432,2835,1001,82,857,850,17,031,600700,17031600700,0.152381,0.353086,0.028924,0.302293,0.299824
4,"Census Tract 6119, Cook County, Illinois",1639,24507,1639,766,1639,26,1175,0,438,17,031,611900,17031611900,0.467358,0.015863,0.716901,0.000000,0.267236


In [16]:
fairness_cols = [
    "total_population",
    "median_household_income",
    "poverty_rate",
    "pct_white_non_hispanic",
    "pct_black_non_hispanic",
    "pct_asian_non_hispanic",
    "pct_hispanic"
]

acs_fairness[fairness_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
total_population,1319.0,3.941073e+03,1.878442e+03,0.0,2506.500000,3818.000000,5159.500000,20087.000000
median_household_income,1319.0,-2.459312e+06,4.098792e+07,-666666666.0,42301.000000,60000.000000,86348.000000,250001.000000
poverty_rate,1315.0,1.624773e-01,1.249615e-01,0.0,0.066822,0.127010,0.227102,0.749475
pct_white_non_hispanic,1315.0,3.911069e-01,3.085193e-01,0.0,0.062832,0.378989,0.682237,0.967855
pct_black_non_hispanic,1315.0,2.845208e-01,3.625682e-01,0.0,0.018412,0.058403,0.610188,1.000000
pct_asian_non_hispanic,1315.0,6.462926e-02,9.493854e-02,0.0,0.002706,0.027646,0.085808,0.863344
pct_hispanic,1315.0,2.386868e-01,2.622091e-01,0.0,0.050865,0.124526,0.332129,0.991284


In [17]:
acs_fairness[fairness_cols].isna().sum()

total_population           0
median_household_income    0
poverty_rate               4
pct_white_non_hispanic     4
pct_black_non_hispanic     4
pct_asian_non_hispanic     4
pct_hispanic               4
dtype: int64

In [18]:
for col in fairness_cols:
    print(
        f"{col}: "
        f"min={acs_fairness[col].min():.3f}, "
        f"median={acs_fairness[col].median():.3f}, "
        f"max={acs_fairness[col].max():.3f}"
    )

total_population: min=0.000, median=3818.000, max=20087.000
median_household_income: min=-666666666.000, median=60000.000, max=250001.000
poverty_rate: min=0.000, median=0.127, max=0.749
pct_white_non_hispanic: min=0.000, median=0.379, max=0.968
pct_black_non_hispanic: min=0.000, median=0.058, max=1.000
pct_asian_non_hispanic: min=0.000, median=0.028, max=0.863
pct_hispanic: min=0.000, median=0.125, max=0.991


## 8. Cleaning ACS Special Values

Inspection of the retrieved ACS variables identified a special negative value in
`median_household_income`. This value does not represent an actual household
income; it is an ACS sentinel value indicating that the estimate is unavailable
or not applicable.

Before merging the demographic data with the property records, invalid ACS
sentinel values are converted to missing values. We also inspect tracts with
undefined demographic rates to determine whether they result from zero
population denominators.

In [19]:
# ACS estimates should not contain negative values for any of the
# demographic measures used in this project.
acs_numeric_cols = [
    "total_population",
    "median_household_income",
    "poverty_universe",
    "below_poverty",
    "race_ethnicity_total",
    "white_non_hispanic",
    "black_non_hispanic",
    "asian_non_hispanic",
    "hispanic"
]

for col in acs_numeric_cols:
    acs_fairness.loc[acs_fairness[col] < 0, col] = np.nan

In [20]:
# Because our rates were computed before this cleaning, recalculate them
acs_fairness["poverty_rate"] = (
    acs_fairness["below_poverty"]
    / acs_fairness["poverty_universe"]
)

acs_fairness["pct_white_non_hispanic"] = (
    acs_fairness["white_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_black_non_hispanic"] = (
    acs_fairness["black_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_asian_non_hispanic"] = (
    acs_fairness["asian_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_hispanic"] = (
    acs_fairness["hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

In [21]:
print("Missing values after ACS cleaning:")
print(acs_fairness[fairness_cols].isna().sum())

print("\nTracts with undefined demographic rates:")
acs_fairness.loc[
    acs_fairness[
        [
            "poverty_rate",
            "pct_white_non_hispanic",
            "pct_black_non_hispanic",
            "pct_asian_non_hispanic",
            "pct_hispanic"
        ]
    ].isna().any(axis=1),
    [
        "NAME",
        "Census Tract GEOID",
        "total_population",
        "poverty_universe",
        "race_ethnicity_total",
        "median_household_income"
    ]
]

Missing values after ACS cleaning:
total_population           0
median_household_income    5
poverty_rate               4
pct_white_non_hispanic     4
pct_black_non_hispanic     4
pct_asian_non_hispanic     4
pct_hispanic               4
dtype: int64

Tracts with undefined demographic rates:


,NAME,Census Tract GEOID,total_population,poverty_universe,race_ethnicity_total,median_household_income
484,"Census Tract 9801, Cook County, Illinois",17031980100,0.0,0.0,0.0,NaN
485,"Census Tract 9800, Cook County, Illinois",17031980000,0.0,0.0,0.0,NaN
486,"Census Tract 9900, Cook County, Illinois",17031990000,0.0,0.0,0.0,NaN
845,"Census Tract 3817, Cook County, Illinois",17031381700,0.0,0.0,0.0,NaN


## 9. Merge ACS Demographic Context with Property Records

After validating Census tract identifiers and cleaning ACS special values,
the tract-level demographic variables are merged onto the Cook County property
dataset.

The merge uses `Census Tract GEOID` as the geographic key. Because each ACS
row represents one Census tract while the property dataset may contain many
properties within the same tract, this is a many-to-one merge.

A left join is used so that every property record is retained. Merge integrity
is then evaluated by checking:

- row counts before and after the merge,
- uniqueness of ACS tract identifiers,
- Census variable coverage,
- and the merge indicator.

In [22]:
acs_merge_cols = [
    "Census Tract GEOID",
    "total_population",
    "median_household_income",
    "poverty_rate",
    "pct_white_non_hispanic",
    "pct_black_non_hispanic",
    "pct_asian_non_hispanic",
    "pct_hispanic"
]

acs_for_merge = acs_fairness[acs_merge_cols].copy()

In [23]:
print("ACS rows:", len(acs_for_merge))
print(
    "Unique ACS GEOIDs:",
    acs_for_merge["Census Tract GEOID"].nunique()
)
print(
    "Duplicate ACS GEOIDs:",
    acs_for_merge["Census Tract GEOID"].duplicated().sum()
)

ACS rows: 1319
Unique ACS GEOIDs: 1319
Duplicate ACS GEOIDs: 0


In [24]:
rows_before = len(data)

data_fair = data.merge(
    acs_for_merge,
    on="Census Tract GEOID",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(data_fair)

print("Rows before merge:", rows_before)
print("Rows after merge:", rows_after)
print("Rows added/lost:", rows_after - rows_before)

Rows before merge: 204792
Rows after merge: 204792
Rows added/lost: 0


In [25]:
print("Merge status:")
print(data_fair["_merge"].value_counts())

print("\nMerge percentages:")
print(
    data_fair["_merge"]
    .value_counts(normalize=True)
    .mul(100)
    .round(3)
)

Merge status:
_merge
both          204792
left_only          0
right_only         0
Name: count, dtype: int64

Merge percentages:
_merge
both          100.0
left_only       0.0
right_only      0.0
Name: proportion, dtype: float64


In [26]:
fairness_variables = [
    "median_household_income",
    "poverty_rate",
    "pct_white_non_hispanic",
    "pct_black_non_hispanic",
    "pct_asian_non_hispanic",
    "pct_hispanic"
]

property_missing = (
    data_fair[fairness_variables]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print("Property-level missingness (%):")
print(property_missing.round(4))

Property-level missingness (%):
median_household_income    0.0024
poverty_rate               0.0000
pct_white_non_hispanic     0.0000
pct_black_non_hispanic     0.0000
pct_asian_non_hispanic     0.0000
pct_hispanic               0.0000
dtype: float64


In [27]:
print(
    "Unique property tracts after merge:",
    data_fair["Census Tract GEOID"].nunique()
)

print(
    "Properties with ACS tract match:",
    (data_fair["_merge"] == "both").sum()
)

print(
    "Properties without ACS tract match:",
    (data_fair["_merge"] == "left_only").sum()
)

Unique property tracts after merge: 1265
Properties with ACS tract match: 204792
Properties without ACS tract match: 0


In [28]:
data_fair[
    [
        "Census Tract GEOID",
        "median_household_income",
        "poverty_rate",
        "pct_white_non_hispanic",
        "pct_black_non_hispanic",
        "pct_asian_non_hispanic",
        "pct_hispanic",
        "_merge"
    ]
].head()

,Census Tract GEOID,median_household_income,poverty_rate,pct_white_non_hispanic,pct_black_non_hispanic,pct_asian_non_hispanic,pct_hispanic,_merge
0,17031600600,37670.0,0.142176,0.195611,0.012405,0.456107,0.335878,both
1,17031200100,53718.0,0.052053,0.207247,0.024505,0.043274,0.709854,both
2,17031491400,33290.0,0.338778,0.022182,0.961031,0.000000,0.001499,both
3,17031810301,79850.0,0.141603,0.340032,0.341161,0.039964,0.261684,both
4,17031830300,38043.0,0.140791,0.240107,0.741113,0.013637,0.004918,both


### ACS–Property Merge Validation

The ACS demographic data were successfully merged with the Cook County property
dataset using `Census Tract GEOID` as the geographic key.

The ACS dataset contains 1,319 rows representing 1,319 unique Census tracts,
with no duplicated GEOIDs. This confirms that the ACS table satisfies the
many-to-one merge assumption: multiple property observations may belong to the
same Census tract, while each tract appears only once in the ACS dataset.

The property dataset contained 204,792 observations before the merge and
204,792 observations afterward. No rows were added or lost, indicating that the
merge preserved the original modeling sample without duplicating or dropping
properties.

All 204,792 property observations successfully matched an ACS Census tract.
The dataset contains 1,265 unique property tracts, and all 1,265 were represented
in the ACS data. Therefore, both the tract-level and property-level geographic
match rates are 100%.

Coverage of the demographic variables is also nearly complete. Poverty rate and
all racial/ethnic composition variables have 0% missingness among the property
observations. Median household income has only 0.0024% missingness, corresponding
to approximately five property records. Because this missingness is negligible,
these observations can be retained in the overall dataset and excluded only
from analyses that specifically require household income.

Overall, the Census integration is successful. The resulting analysis dataset
contains 204,792 property observations across 1,265 Census tracts with complete
geographic linkage and nearly complete demographic coverage. This dataset can
therefore support tract-level socioeconomic and demographic fairness analyses
without materially reducing the original property sample.

## 10. Distribution of Demographic Context Across Property Observations

The ACS variables were initially examined at the Census-tract level, where each
tract contributed equally to the distribution. After merging the ACS information
with the property dataset, the relevant distribution for model evaluation is the
distribution across individual property observations.

Because different Census tracts contribute different numbers of properties to
the modeling dataset, the demographic distribution among properties may differ
from the distribution among Census tracts.

Before defining fairness comparison groups, we therefore examine:

1. the overall distribution of each demographic variable,
2. important quantiles of each variable,
3. the number of property observations contributed by each Census tract, and
4. differences between tract-level and property-level demographic medians.

In [29]:
property_fairness_summary = (
    data_fair[fairness_variables]
    .describe()
    .T
)

property_fairness_summary

,count,mean,std,min,25%,50%,75%,max
median_household_income,204787.0,78734.896439,38250.907339,11146.000000,52381.000000,70497.000000,95897.000000,250001.000000
poverty_rate,204792.0,0.114121,0.090905,0.002958,0.047677,0.084902,0.155753,0.662912
pct_white_non_hispanic,204792.0,0.454145,0.311908,0.000000,0.127812,0.497684,0.742903,0.967855
pct_black_non_hispanic,204792.0,0.244973,0.346704,0.000000,0.011843,0.039630,0.440998,1.000000
pct_asian_non_hispanic,204792.0,0.055463,0.077927,0.000000,0.005082,0.023805,0.069496,0.863344
pct_hispanic,204792.0,0.225671,0.242240,0.000000,0.049413,0.126185,0.310860,0.991284


In [30]:
quantiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]

for col in fairness_variables:
    print(f"\n{col}")
    print(
        data_fair[col]
        .quantile(quantiles)
        .round(4)
    )


median_household_income
0.05     33423.0
0.10     40426.0
0.25     52381.0
0.50     70497.0
0.75     95897.0
0.90    126250.0
0.95    149722.0
Name: median_household_income, dtype: float64

poverty_rate
0.05    0.0194
0.10    0.0270
0.25    0.0477
0.50    0.0849
0.75    0.1558
0.90    0.2500
0.95    0.2953
Name: poverty_rate, dtype: float64

pct_white_non_hispanic
0.05    0.0058
0.10    0.0188
0.25    0.1278
0.50    0.4977
0.75    0.7429
0.90    0.8530
0.95    0.8899
Name: pct_white_non_hispanic, dtype: float64

pct_black_non_hispanic
0.05    0.0005
0.10    0.0030
0.25    0.0118
0.50    0.0396
0.75    0.4410
0.90    0.9057
0.95    0.9621
Name: pct_black_non_hispanic, dtype: float64

pct_asian_non_hispanic
0.05    0.0000
0.10    0.0000
0.25    0.0051
0.50    0.0238
0.75    0.0695
0.90    0.1595
0.95    0.2331
Name: pct_asian_non_hispanic, dtype: float64

pct_hispanic
0.05    0.0046
0.10    0.0145
0.25    0.0494
0.50    0.1262
0.75    0.3109
0.90    0.6296
0.95    0.7859
Name: pct_hispa

In [31]:
tract_property_counts = (
    data_fair["Census Tract GEOID"]
    .value_counts()
)

print("Properties per tract:")
print(tract_property_counts.describe())

print("\nLargest 10 tracts by property count:")
print(tract_property_counts.head(10))

Properties per tract:
count    1265.000000
mean      161.890909
std       142.849585
min         1.000000
25%        42.000000
50%       121.000000
75%       249.000000
max       749.000000
Name: count, dtype: float64

Largest 10 tracts by property count:
Census Tract GEOID
17031804201    749
17031830400    695
17031834300    654
17031804310    628
17031815500    618
17031804202    606
17031826202    599
17031804306    593
17031171000    590
17031640300    583
Name: count, dtype: int64


In [32]:
comparison = pd.DataFrame({
    "tract_level_median":
        acs_fairness[fairness_variables].median(),

    "property_level_median":
        data_fair[fairness_variables].median()
})

comparison["difference"] = (
    comparison["property_level_median"]
    - comparison["tract_level_median"]
)

comparison

,tract_level_median,property_level_median,difference
median_household_income,60161.500000,70497.000000,10335.500000
poverty_rate,0.127010,0.084902,-0.042108
pct_white_non_hispanic,0.378989,0.497684,0.118695
pct_black_non_hispanic,0.058403,0.039630,-0.018773
pct_asian_non_hispanic,0.027646,0.023805,-0.003841
pct_hispanic,0.124526,0.126185,0.001658


### Property-Level Fairness Variable Distributions

The demographic context represented in the property sample differs meaningfully
from the distribution across Cook County Census tracts as a whole.

At the property level, the median household income is approximately $70,497$,
compared with about $60,162$ at the tract level. The property-level median is
therefore roughly $10,336$ higher. Similarly, the median poverty rate among
property observations is about 8.5%, compared with about 12.7% across Census
tracts.

This indicates that the property dataset is relatively more concentrated in
higher-income and lower-poverty neighborhoods than Cook County tracts overall.

The racial and ethnic composition also differs between the two weighting
schemes. The property-level median non-Hispanic White share is approximately
49.8%, compared with about 37.9% at the tract level. In contrast, the
property-level median non-Hispanic Black share is approximately 4.0%, compared
with about 5.8% across tracts. Hispanic composition is similar under the two
weighting schemes, with a property-level median of approximately 12.6% and a
tract-level median of approximately 12.5%.

These differences demonstrate why fairness groups should be defined using the
distribution of the actual modeling sample rather than the unweighted
distribution of all Census tracts.

The socioeconomic variables also show substantial variation within the property
sample. Median household income ranges from approximately $11,146$ to $250,001$,
with an interquartile range from about $52,381$ to $95,897$. Poverty rates range
from approximately 0.3% to 66.3%, with an interquartile range from about 4.8%
to 15.6%.

The racial and ethnic composition variables are strongly heterogeneous across
the sample. For example, the non-Hispanic Black share has a median of only about
4.0%, but its 75th percentile is approximately 44.1% and its 90th percentile is
approximately 90.6%. The Hispanic share similarly increases from a median of
approximately 12.6% to about 63.0% at the 90th percentile. These distributions
are highly skewed, so simple equal-width thresholds would not produce balanced
comparison groups.

Property observations are also unevenly distributed across Census tracts. The
1,265 represented tracts contain an average of approximately 162 property
observations, with a median of 121. The interquartile range is approximately
42 to 249 observations per tract, while the largest tract contains 749
observations. Although some tracts contribute substantially more observations
than others, no single tract appears large enough to dominate the full sample of
204,792 properties.

Overall, these results support defining fairness groups from property-level
quantiles rather than arbitrary fixed thresholds. Quantile-based grouping will
create sufficiently populated comparison groups while respecting the strongly
skewed distributions of income, poverty, and neighborhood demographic
composition.

## 11. Constructing Fairness Comparison Groups

The property-level demographic distributions are highly heterogeneous and, for
several variables, strongly skewed. Fixed-width thresholds would therefore
produce comparison groups with very different sample sizes.

To support stable comparisons of model performance, fairness groups are defined
using property-level quantiles. This gives each group approximately the same
number of property observations and avoids selecting arbitrary demographic
thresholds.

For socioeconomic context, properties are divided into quartiles based on:

- median household income, and
- Census-tract poverty rate.

For racial and ethnic composition, quartile groups are also constructed
initially. These groups should be interpreted as neighborhood-composition
categories rather than individual racial or ethnic identities.

In [33]:
data_fair["income_group"] = pd.qcut(
    data_fair["median_household_income"],
    q=4,
    labels=[
        "Q1: Lowest income",
        "Q2",
        "Q3",
        "Q4: Highest income"
    ]
)

In [34]:
data_fair["poverty_group"] = pd.qcut(
    data_fair["poverty_rate"],
    q=4,
    labels=[
        "Q1: Lowest poverty",
        "Q2",
        "Q3",
        "Q4: Highest poverty"
    ]
)

In [35]:
composition_variables = {
    "pct_white_non_hispanic": "white_composition_group",
    "pct_black_non_hispanic": "black_composition_group",
    "pct_asian_non_hispanic": "asian_composition_group",
    "pct_hispanic": "hispanic_composition_group"
}

for variable, new_col in composition_variables.items():
    data_fair[new_col] = pd.qcut(
        data_fair[variable],
        q=4,
        labels=[
            "Q1: Lowest share",
            "Q2",
            "Q3",
            "Q4: Highest share"
        ],
        duplicates="drop"
    )

In [36]:
group_cols = [
    "income_group",
    "poverty_group",
    "white_composition_group",
    "black_composition_group",
    "asian_composition_group",
    "hispanic_composition_group"
]

for col in group_cols:
    print(f"\n{col}")
    print(data_fair[col].value_counts(dropna=False).sort_index())


income_group
income_group
Q1: Lowest income     51341
Q2                    51336
Q3                    50920
Q4: Highest income    51190
NaN                       5
Name: count, dtype: int64

poverty_group
poverty_group
Q1: Lowest poverty     51472
Q2                     50992
Q3                     51292
Q4: Highest poverty    51036
Name: count, dtype: int64

white_composition_group
white_composition_group
Q1: Lowest share     51258
Q2                   51190
Q3                   51147
Q4: Highest share    51197
Name: count, dtype: int64

black_composition_group
black_composition_group
Q1: Lowest share     51346
Q2                   51241
Q3                   51360
Q4: Highest share    50845
Name: count, dtype: int64

asian_composition_group
asian_composition_group
Q1: Lowest share     51464
Q2                   51246
Q3                   50889
Q4: Highest share    51193
Name: count, dtype: int64

hispanic_composition_group
hispanic_composition_group
Q1: Lowest share     51559
Q2   

In [37]:
group_variable_pairs = [
    ("income_group", "median_household_income"),
    ("poverty_group", "poverty_rate"),
    ("white_composition_group", "pct_white_non_hispanic"),
    ("black_composition_group", "pct_black_non_hispanic"),
    ("asian_composition_group", "pct_asian_non_hispanic"),
    ("hispanic_composition_group", "pct_hispanic")
]

for group_col, value_col in group_variable_pairs:
    print(f"\n{group_col}")

    print(
        data_fair.groupby(
            group_col,
            observed=True
        )[value_col]
        .agg(["count", "min", "median", "max"])
    )


income_group
                    count      min    median       max
income_group                                          
Q1: Lowest income   51341  11146.0   43143.0   52381.0
Q2                  51336  52415.0   61342.0   70497.0
Q3                  50920  70505.0   81389.0   95897.0
Q4: Highest income  51190  96194.0  117500.0  250001.0

poverty_group
                     count       min    median       max
poverty_group                                           
Q1: Lowest poverty   51472  0.002958  0.030228  0.047677
Q2                   50992  0.047842  0.065363  0.084902
Q3                   51292  0.085106  0.109709  0.155753
Q4: Highest poverty  51036  0.156182  0.230978  0.662912

white_composition_group
                         count       min    median       max
white_composition_group                                     
Q1: Lowest share         51258  0.000000  0.027449  0.127812
Q2                       51190  0.128659  0.325255  0.497684
Q3                       51147

### Fairness Group Construction

Quantile-based fairness groups were successfully created for neighborhood income,
poverty, and racial/ethnic composition.

The socioeconomic groups are well balanced. The four income groups each contain
approximately 51,000 property observations, with only five observations missing
because median household income was unavailable for a very small number of
properties. The income quartiles also correspond to clearly separated ranges:

- Q1: approximately 11,146 to 52,381
- Q2: approximately 52,415 to 70,497
- Q3: approximately 70,505 to 95,897
- Q4: approximately 96,194 to 250,001

The poverty quartiles are similarly balanced, with roughly 51,000 observations
per group. Their ranges increase monotonically from approximately 0.3% to 4.8%
poverty in Q1, up to approximately 15.6% to 66.3% in Q4.

The demographic composition groups also remain close to equal in size despite
the strong skew observed in the underlying variables. This confirms that the
quantile-based approach is more appropriate than fixed-width thresholds for this
dataset.

The non-Hispanic Black composition groups are especially informative because the
distribution is highly polarized. The first two quartiles contain tracts with
very low Black population shares, while Q3 spans roughly 4.0% to 44.1%, and Q4
ranges from approximately 44.2% to 100%. This large jump confirms that the
underlying distribution is strongly skewed rather than approximately uniform.

The Asian composition variable also shows substantial concentration near zero.
The median share within Q1 is 0%, while the highest-share quartile begins around
7.0% and extends to approximately 86.3%. Hispanic and non-Hispanic White
composition groups show similarly wide variation across quartiles.

Overall, the fairness groups are sufficiently balanced for later model-error
comparisons. The quartile construction also preserves meaningful differences in
neighborhood socioeconomic and demographic context without relying on arbitrary
fixed thresholds.

These groups should be interpreted as neighborhood-level contextual categories.
For example, `Q4: Highest share` for the Black composition variable refers to
properties located in Census tracts with relatively high non-Hispanic Black
population shares; it does not describe the race or ethnicity of individual
property owners.

## 12. Fairness Evaluation Framework

Fairness is evaluated after model prediction by comparing model errors across the
predefined neighborhood groups.

The analysis distinguishes between overall predictive accuracy and systematic
error direction. A model may have similar average accuracy across groups while
still consistently overpredicting values in one group and underpredicting values
in another.

The primary group-level evaluation metrics are:

1. **Mean Absolute Error (MAE)**  
   Measures the average absolute prediction error in dollars.

2. **Mean Absolute Percentage Error (MAPE)**  
   Measures prediction error relative to the observed property value, allowing
   comparison across properties with different price levels.

3. **Overprediction Rate**  
   Measures the proportion of observations for which predicted value exceeds
   observed value. This is particularly relevant for assessment fairness because
   systematic overvaluation may translate into disproportionate tax burdens.

4. **Mean Signed Percentage Error (MSPE)**  
   Measures both the magnitude and direction of systematic prediction error.
   Positive values indicate average overprediction, while negative values indicate
   average underprediction.

Fairness will be examined by comparing these metrics across income, poverty, and
racial/ethnic composition groups. Demographic variables are treated as
evaluation attributes rather than individual-level characteristics.

In [38]:
def add_prediction_errors(df, actual_col, predicted_col):
    """
    Add property-level prediction error measures.
    """
    result = df.copy()

    result["error"] = (
        result[predicted_col] - result[actual_col]
    )

    result["absolute_error"] = result["error"].abs()

    result["percentage_error"] = (
        result["error"] / result[actual_col]
    )

    result["absolute_percentage_error"] = (
        result["percentage_error"].abs()
    )

    result["overpredicted"] = (
        result[predicted_col] > result[actual_col]
    )

    return result

In [39]:
def fairness_summary(df, group_col):
    """
    Summarize model error across fairness groups.
    """
    return (
        df.groupby(group_col, observed=True)
        .agg(
            n=("error", "size"),
            mae=("absolute_error", "mean"),
            mape=("absolute_percentage_error", "mean"),
            overprediction_rate=("overpredicted", "mean"),
            mean_signed_percentage_error=("percentage_error", "mean")
        )
    )

# Part II: Feature Engineering and Baseline Model

## 1. Modeling Setup and Baseline Prediction Framework

With the fairness evaluation framework defined, the next stage is to construct
the housing-price prediction task.

The modeling pipeline is designed so that:

- property characteristics are used as predictive features,
- Census demographic variables are reserved primarily for post-model fairness
  evaluation,
- the target variable represents observed property value or sale price,
- training and test data are separated before model evaluation,
- and baseline performance is established before fitting more flexible models.

The first modeling therefore focuses on:

1. identifying the prediction target,
2. reviewing candidate predictors,
3. separating predictive features from fairness attributes,
4. checking missingness and data types,
5. and defining the train/test split.

In [40]:
print("Dataset shape:", data_fair.shape)

print("\nColumns:")
for col in data_fair.columns:
    print(col)

Dataset shape: (204792, 79)

Columns:
Unnamed: 0
PIN
Property Class
Neighborhood Code
Land Square Feet
Town Code
Apartments
Wall Material
Roof Material
Basement
Basement Finish
Central Heating
Other Heating
Central Air
Fireplaces
Attic Type
Attic Finish
Design Plan
Cathedral Ceiling
Construction Quality
Site Desirability
Garage 1 Size
Garage 1 Material
Garage 1 Attachment
Garage 1 Area
Garage 2 Size
Garage 2 Material
Garage 2 Attachment
Garage 2 Area
Porch
Other Improvements
Building Square Feet
Repair Condition
Multi Code
Number of Commercial Units
Estimate (Land)
Estimate (Building)
Deed No.
Sale Price
Longitude
Latitude
Census Tract
Multi Property Indicator
Modeling Group
Age
Use
O'Hare Noise
Floodplain
Road Proximity
Sale Year
Sale Quarter
Sale Half-Year
Sale Quarter of Year
Sale Month of Year
Sale Half of Year
Most Recent Sale
Age Decade
Pure Market Filter
Garage Indicator
Neigborhood Code (mapping)
Town and Neighborhood
Description
Lot Size
Census Tract Code
Census Tract GEOID
to

In [41]:
candidate_target_cols = [
    col for col in data_fair.columns
    if any(
        keyword in col.lower()
        for keyword in ["price", "sale", "market", "value"]
    )
]

print("Potential target columns:")
print(candidate_target_cols)

Potential target columns:
['Sale Price', 'Sale Year', 'Sale Quarter', 'Sale Half-Year', 'Sale Quarter of Year', 'Sale Month of Year', 'Sale Half of Year', 'Most Recent Sale', 'Pure Market Filter']


In [43]:
numeric_cols = data_fair.select_dtypes(include="number").columns.tolist()
categorical_cols = data_fair.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

print("\nNumeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Number of numeric columns: 68
Number of categorical columns: 10

Numeric columns:
['Unnamed: 0', 'PIN', 'Property Class', 'Neighborhood Code', 'Land Square Feet', 'Town Code', 'Apartments', 'Wall Material', 'Roof Material', 'Basement', 'Basement Finish', 'Central Heating', 'Other Heating', 'Central Air', 'Fireplaces', 'Attic Type', 'Attic Finish', 'Design Plan', 'Cathedral Ceiling', 'Construction Quality', 'Site Desirability', 'Garage 1 Size', 'Garage 1 Material', 'Garage 1 Attachment', 'Garage 1 Area', 'Garage 2 Size', 'Garage 2 Material', 'Garage 2 Attachment', 'Garage 2 Area', 'Porch', 'Other Improvements', 'Building Square Feet', 'Repair Condition', 'Multi Code', 'Number of Commercial Units', 'Estimate (Land)', 'Estimate (Building)', 'Deed No.', 'Sale Price', 'Longitude', 'Latitude', 'Census Tract', 'Multi Property Indicator', 'Age', 'Use', "O'Hare Noise", 'Floodplain', 'Road Proximity', 'Sale Year', 'Sale Quarter', 'Sale Half-Year', 'Sale Quarter of Year', 'Sale Month of Year', 'S

/tmp/ipykernel_91517/1987528689.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = data_fair.select_dtypes(


In [44]:
fairness_only_cols = [
    "median_household_income",
    "poverty_rate",
    "pct_white_non_hispanic",
    "pct_black_non_hispanic",
    "pct_asian_non_hispanic",
    "pct_hispanic",
    "income_group",
    "poverty_group",
    "white_composition_group",
    "black_composition_group",
    "asian_composition_group",
    "hispanic_composition_group"
]

In [45]:
metadata_cols = [
    "Census Tract GEOID",
    "_merge"
]

In [46]:
missing_summary = (
    data_fair
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(
    missing_summary[
        missing_summary > 0
    ].head(30)
)

median_household_income    0.002442
income_group               0.002442
dtype: float64


In [47]:
leakage_keywords = [
    "price",
    "sale",
    "market",
    "assessed",
    "assessment",
    "value"
]

possible_leakage_cols = [
    col for col in data_fair.columns
    if any(
        keyword in col.lower()
        for keyword in leakage_keywords
    )
]

print("Columns requiring leakage review:")
for col in possible_leakage_cols:
    print(col)

Columns requiring leakage review:
Sale Price
Sale Year
Sale Quarter
Sale Half-Year
Sale Quarter of Year
Sale Month of Year
Sale Half of Year
Most Recent Sale
Pure Market Filter


## Target, Missingness, and Leakage Review

The merged modeling dataset contains 204,792 observations and 79 columns.
Missingness is extremely limited: only `median_household_income` and the derived
`income_group` contain missing values, each for approximately 0.0024% of the
sample. Because these Census variables are reserved for fairness evaluation,
missing predictor values are not currently a major preprocessing concern.

`Sale Price` is the appropriate prediction target. The codebook explicitly
defines this variable as the observed sale price of the property.

The column data types require additional interpretation. Although most variables
are stored numerically, many represent coded categories rather than continuous
quantities. Examples include property class, wall material, basement type,
heating type, garage characteristics, repair condition, and property use.
Therefore, numeric storage type alone should not determine how a variable is
treated during modeling.

The initial keyword-based leakage search identified several sale-related
variables, but not every variable containing the word "Sale" is target leakage.
Variables such as `Sale Year`, `Sale Quarter`, and `Sale Month of Year` describe
when a transaction occurred and may legitimately capture changes in the housing
market over time. In contrast, `Most Recent Sale` depends on whether another sale
occurs later and therefore contains information that would not necessarily be
known at the time of the transaction. It should be excluded from prediction.

`Pure Market Filter` is an indicator identifying pure-market transactions.
Rather than using it as a predictor, it should be treated as a sample-selection
variable if the modeling population is restricted to arm's-length market sales.

The prior-year `Estimate (Land)` and `Estimate (Building)` variables require a
special policy decision. Because they represent Board of Review estimated market
values from the prior tax year, they are not direct target leakage. However,
including existing assessment estimates in the primary model would make the new
price model partly dependent on the assessment system whose potential inequities
we ultimately want to evaluate. For this reason, they should be excluded from
the primary fairness model. They could later be introduced in a separate
benchmark model to measure how much predictive accuracy is gained by relying on
existing assessments.

Identifier variables such as `PIN`, `Deed No.`, and the imported `Unnamed: 0`
index should also be excluded because their numerical values do not represent
meaningful quantitative property characteristics.

Finally, several fields are poor candidates for the primary feature set based on
the dataset documentation. `Construction Quality` contains unusual coding and is
described as generally not useful analytically; `Site Desirability` has
insufficient variation; and `Other Improvements` contains ambiguous codes.
These variables should initially be excluded rather than allowing noisy or
poorly documented information to influence the baseline model.

Overall, the dataset is technically ready for modeling, but the final modeling
population should be established before creating a train/test split. In
particular, property-type composition, pure-market-sale status, target
distribution, and repeated sales of the same PIN must first be examined.

In [49]:
target_col = "Sale Price"
fairness_only_cols = [
    "total_population",
    "median_household_income",
    "poverty_rate",
    "pct_white_non_hispanic",
    "pct_black_non_hispanic",
    "pct_asian_non_hispanic",
    "pct_hispanic",
    "income_group",
    "poverty_group",
    "white_composition_group",
    "black_composition_group",
    "asian_composition_group",
    "hispanic_composition_group"
]

In [50]:
identifier_cols = [
    "Unnamed: 0",
    "PIN",
    "Deed No.",
    "Census Tract",
    "Census Tract GEOID"
]

post_sale_or_filter_cols = [
    "Most Recent Sale",
    "Pure Market Filter",
    "_merge"
]

assessment_cols = [
    "Estimate (Land)",
    "Estimate (Building)"
]

low_quality_cols = [
    "Construction Quality",
    "Site Desirability",
    "Other Improvements"
]

In [51]:
print("Modeling Group counts:")
print(data_fair["Modeling Group"].value_counts(dropna=False))

print("\nModeling Group percentages:")
print(
    data_fair["Modeling Group"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

Modeling Group counts:
Modeling Group
SF    204792
Name: count, dtype: int64

Modeling Group percentages:
Modeling Group
SF    100.0
Name: proportion, dtype: float64


In [53]:
print("Pure Market Filter:")
print(data_fair["Pure Market Filter"].value_counts(dropna=False))

print("\nPercentages:")
print(
    data_fair["Pure Market Filter"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

Pure Market Filter:
Pure Market Filter
1    167384
0     37408
Name: count, dtype: int64

Percentages:
Pure Market Filter
1    81.73
0    18.27
Name: proportion, dtype: float64


In [54]:
print("Sale Price summary:")
print(data_fair["Sale Price"].describe())

print("\nSale Price quantiles:")
print(
    data_fair["Sale Price"].quantile(
        [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1]
    )
)

Sale Price summary:
count    2.047920e+05
mean     2.451646e+05
std      3.628694e+05
min      1.000000e+00
25%      4.520000e+04
50%      1.750000e+05
75%      3.120000e+05
max      7.100000e+07
Name: Sale Price, dtype: float64

Sale Price quantiles:
0.00           1.0
0.01           1.0
0.05           1.0
0.25       45200.0
0.50      175000.0
0.75      312000.0
0.95      775000.0
0.99     1500000.0
1.00    71000000.0
Name: Sale Price, dtype: float64


In [55]:
print("Sale Price <= 0:",
      (data_fair["Sale Price"] <= 0).sum())

print("Sale Price == 1:",
      (data_fair["Sale Price"] == 1).sum())

Sale Price <= 0: 0
Sale Price == 1: 35546


In [56]:
sales_per_pin = data_fair["PIN"].value_counts()

print("Unique PINs:", data_fair["PIN"].nunique())
print("Total observations:", len(data_fair))

print("\nSales per PIN:")
print(sales_per_pin.describe())

print(
    "\nPINs appearing more than once:",
    (sales_per_pin > 1).sum()
)

print(
    "Observations belonging to repeated PINs:",
    data_fair["PIN"].isin(
        sales_per_pin[sales_per_pin > 1].index
    ).sum()
)

Unique PINs: 167191
Total observations: 204792

Sales per PIN:
count    167191.000000
mean          1.224898
std           0.494788
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          14.000000
Name: count, dtype: float64

PINs appearing more than once: 32432
Observations belonging to repeated PINs: 70033


In [57]:
print("Sales by year:")
print(
    data_fair["Sale Year"]
    .value_counts()
    .sort_index()
)

Sales by year:
Sale Year
2013    23990
2014    22832
2015    23633
2016    31751
2017    33053
2018    31770
2019    37763
Name: count, dtype: int64


### Modeling Population and Split Diagnostics

The modeling-population diagnostics clarify the appropriate scope of the housing
price prediction problem.

All 204,792 observations belong to the `SF` modeling group. Therefore, the
dataset is already restricted to single-family properties, and there is no need
to construct separate models for single-family, multi-family, and no-characteristic
properties. This provides a relatively consistent property type for the primary
prediction task.

The `Pure Market Filter` indicates that 167,384 observations, or approximately
81.7% of the dataset, are classified as pure-market transactions. The remaining
37,408 observations, or approximately 18.3%, are not. Because the goal is to
predict economically meaningful market prices, the primary modeling sample should
ultimately be restricted to pure-market transactions rather than using this
indicator as a predictive feature.

The sale-price distribution confirms the importance of this restriction. Although
the median sale price is 175,000 and the 75th percentile is 312,000, the lower
tail contains a substantial number of nominal transactions. In particular,
35,546 observations have a recorded sale price of exactly $1. These values are
not economically meaningful market prices and would severely distort both model
training and percentage-based error metrics such as MAPE.

At the upper end, the sale-price distribution is strongly right-skewed. The 95th
percentile is approximately 775,000, the 99th percentile is approximately
1.5 million, and the maximum recorded sale price is $71 million. Therefore,
even after removing non-market transactions, the target distribution should be
examined again before deciding whether additional treatment of extreme values or
a log-transformed target is appropriate.

Repeated property sales are also common. The 204,792 observations correspond to
167,191 unique PINs. A total of 32,432 properties appear more than once, and
70,033 observations belong to properties represented multiple times. Thus,
approximately one-third of the observations come from properties that appear
elsewhere in the dataset.

This finding has implications for train/test design. A purely random row split
could place different sales of the same property in both the training and test
sets, producing an overly optimistic estimate of model generalization. The
dataset also has a natural temporal structure from 2013 through 2019, with
substantial observations in every year and 37,763 transactions in 2019.

For this reason, a temporal holdout is preferable to a conventional random
split. Earlier years can be used for training while 2019 serves as the primary
test period. This more closely represents the practical task of using historical
property information to predict prices for future transactions.

Before finalizing this split, however, the relationship between the `Pure Market
Filter`, the $1 transactions, and the remaining sale-price distribution should be
verified. This will determine the final modeling sample and whether additional
target filtering is necessary.

In [58]:
sale_one_market_table = pd.crosstab(
    data_fair["Sale Price"] == 1,
    data_fair["Pure Market Filter"],
    margins=True
)

sale_one_market_table

Pure Market Filter,0,1,All
Sale Price,,,
False,1862,167384,169246
True,35546,0,35546
All,37408,167384,204792


In [59]:
print("Pure-market $1 sales:",
      (
          (data_fair["Sale Price"] == 1)
          & (data_fair["Pure Market Filter"] == 1)
      ).sum())

print("Non-pure-market $1 sales:",
      (
          (data_fair["Sale Price"] == 1)
          & (data_fair["Pure Market Filter"] == 0)
      ).sum())

Pure-market $1 sales: 0
Non-pure-market $1 sales: 35546


In [60]:
model_data = (
    data_fair.loc[
        data_fair["Pure Market Filter"] == 1
    ]
    .copy()
)

print("Original observations:", len(data_fair))
print("Pure-market observations:", len(model_data))
print(
    "Observations removed:",
    len(data_fair) - len(model_data)
)

Original observations: 204792
Pure-market observations: 167384
Observations removed: 37408


In [61]:
print("Pure-market Sale Price summary:")
print(model_data["Sale Price"].describe())

print("\nPure-market Sale Price quantiles:")
print(
    model_data["Sale Price"].quantile(
        [0, 0.01, 0.05, 0.10, 0.25,
         0.50, 0.75, 0.90, 0.95, 0.99, 1]
    )
)

print(
    "\nPure-market $1 sales:",
    (model_data["Sale Price"] == 1).sum()
)

Pure-market Sale Price summary:
count    1.673840e+05
mean     2.990826e+05
std      3.281287e+05
min      1.000300e+04
25%      1.250000e+05
50%      2.175000e+05
75%      3.540000e+05
max      8.490078e+06
Name: Sale Price, dtype: float64

Pure-market Sale Price quantiles:
0.00      10003.0
0.01      16000.0
0.05      32694.0
0.10      53200.0
0.25     125000.0
0.50     217500.0
0.75     354000.0
0.90     601402.2
0.95     850000.0
0.99    1600000.0
1.00    8490078.0
Name: Sale Price, dtype: float64

Pure-market $1 sales: 0


In [62]:
model_data.nlargest(
    20,
    "Sale Price"
)[
    [
        "PIN",
        "Sale Price",
        "Sale Year",
        "Land Square Feet",
        "Building Square Feet",
        "Property Class",
        "Neighborhood Code"
    ]
]

,PIN,Sale Price,Sale Year,Land Square Feet,Building Square Feet,Property Class,Neighborhood Code
157521,5161060740000,8490078,2018,71715.685936,6888.0,209,171
194905,5081010510000,8000000,2013,41948.000000,7018.0,209,171
8043,9351120170000,7800000,2015,5850.000000,2772.0,278,150
117989,17031000110000,7400000,2014,5500.000000,7884.0,209,22
193635,5211040010000,7144704,2013,62962.705122,5388.0,209,171
10961,5274140030000,7000000,2013,29100.000000,3616.0,206,171
68886,5274040100000,7000000,2015,26500.000000,4934.0,204,171
74846,5064040400000,7000000,2014,46000.000000,9692.0,209,171
97707,14333031630000,6707000,2018,4100.000000,6082.0,209,12
121800,5211140020000,6650000,2016,38507.000000,6087.0,209,171


In [63]:
for cutoff in [1_000_000, 2_000_000, 5_000_000, 10_000_000]:
    n = (model_data["Sale Price"] > cutoff).sum()
    pct = n / len(model_data) * 100

    print(
        f"Sale Price > ${cutoff:,}: "
        f"{n:,} ({pct:.3f}%)"
    )

Sale Price > $1,000,000: 5,758 (3.440%)
Sale Price > $2,000,000: 838 (0.501%)
Sale Price > $5,000,000: 27 (0.016%)
Sale Price > $10,000,000: 0 (0.000%)


In [64]:
print("Pure-market sales by year:")
print(
    model_data["Sale Year"]
    .value_counts()
    .sort_index()
)

Pure-market sales by year:
Sale Year
2013    23602
2014    22523
2015    20521
2016    25100
2017    26132
2018    25292
2019    24214
Name: count, dtype: int64


In [65]:
train_candidate = model_data[
    model_data["Sale Year"] < 2019
]

test_candidate = model_data[
    model_data["Sale Year"] == 2019
]

train_pins = set(train_candidate["PIN"])
test_pins = set(test_candidate["PIN"])

overlapping_pins = train_pins & test_pins

print("Candidate training observations:", len(train_candidate))
print("Candidate 2019 test observations:", len(test_candidate))

print(
    "2019 test PINs previously observed in training:",
    len(overlapping_pins)
)

print(
    "2019 test observations whose PIN appeared earlier:",
    test_candidate["PIN"].isin(train_pins).sum()
)

print(
    "Percentage of 2019 test observations with prior PIN:",
    round(
        test_candidate["PIN"].isin(train_pins).mean() * 100,
        2
    )
)

Candidate training observations: 143170
Candidate 2019 test observations: 24214
2019 test PINs previously observed in training: 5054
2019 test observations whose PIN appeared earlier: 5153
Percentage of 2019 test observations with prior PIN: 21.28


### Final Modeling Population and Temporal Holdout

The pure-market filter resolves the abnormal lower tail of the sale-price
distribution without requiring an arbitrary minimum-price threshold.

All 35,546 observations with a recorded sale price of 1 are classified as
non-pure-market transactions. None of the 167,384 pure-market observations has
a 1 sale price. After restricting the sample to `Pure Market Filter == 1`,
the minimum sale price becomes 10,003, indicating that the filter successfully
removes the nominal transactions that would otherwise distort model fitting and
percentage-based error measures.

The resulting pure-market sample contains 167,384 single-family property sales.
Its median sale price is 217,500, with an interquartile range from approximately
125,000 to 354,000. The target remains strongly right-skewed: the 90th
percentile is approximately 601,402, the 95th percentile is 850,000, and the
99th percentile is $1.6 million.

The high-price observations do not appear to justify an arbitrary upper-price
cutoff. Approximately 3.44% of pure-market sales exceed 1 million, 0.50%
exceed 2 million, and only 0.016% exceed 5 million. No pure-market sale exceeds
10 million, and the maximum is approximately $8.49 million. Inspection of the
largest transactions shows property characteristics consistent with very large
or high-value homes rather than an obvious repeated sentinel value or processing
error. These observations will therefore be retained.

Because the price distribution remains substantially right-skewed, the primary
modeling target will use a logarithmic transformation of sale price during model
training. Predictions will be transformed back to dollars before computing the
fairness and accuracy metrics. This reduces the disproportionate influence of
the small number of very expensive properties while preserving all legitimate
market transactions.

The pure-market sample also provides substantial observations in every year from
2013 through 2019. There are 143,170 observations from 2013–2018 and 24,214
observations in 2019, supporting a natural temporal holdout in which historical
sales are used to predict future transactions.

However, repeated properties remain important under this split. A total of
5,153 of the 24,214 candidate 2019 test observations, or approximately 21.3%,
have a PIN that was already observed during 2013–2018. Including these
properties in the primary test set could make evaluation easier because the
training data contain an earlier transaction for the same physical property,
even though PIN itself is excluded from the predictor set.

To obtain a stricter estimate of generalization, the primary 2019 test set will
therefore contain only properties whose PIN was not observed during the
2013–2018 training period. The repeated-property 2019 observations can be
retained separately for a secondary analysis if desired.

The final primary modeling design is therefore:

- Population: single-family, pure-market transactions
- Training period: 2013–2018
- Test period: 2019
- Primary test set: 2019 properties not previously observed in training
- Target: log-transformed sale price during fitting
- Evaluation: predictions converted back to dollar values before accuracy and
  fairness metrics are calculated

This design provides a more realistic and conservative evaluation of how well
the model generalizes to future, previously unseen properties.

## 2. Final Modeling Sample and Temporal Train/Test Split

The primary modeling sample is restricted to pure-market single-family sales.
The `Pure Market Filter` is used for sample selection rather than as a predictor.

A temporal evaluation design is used instead of a random train/test split.
Transactions from 2013–2018 form the historical training sample, while 2019
transactions form the future evaluation period.

To avoid evaluating the model on properties that were already represented in the
training period, the primary test set contains only 2019 properties whose PIN was
not observed during 2013–2018.

2019 transactions for previously observed properties are retained separately as
a secondary repeated-property test set.

The model is trained on log-transformed sale price to reduce sensitivity to the
strongly right-skewed target distribution. Predictions are transformed back to
dollars before model accuracy and fairness are evaluated.